In [1]:
using MAT               # pour charger .mat
using LinearAlgebra
using Distributions
using Random
using Symbolics
using ModelingToolkit

In [5]:

@parameters t
D = Differential(t)

# ----------------------------
# Parameters
# ----------------------------
@parameters J1 J2 D1 D2 KL Ks ω0
@parameters α1 α2 β1 β2 Pr1 Pr2

# ----------------------------
# States
# X = [θ1, ω1, Tm1, θ2, ω2, Tm2, N]
# ----------------------------
@variables θ1(t) ω1(t) Tm1(t)
@variables θ2(t) ω2(t) Tm2(t)
@variables N(t)

X = [θ1, ω1, Tm1, θ2, ω2, Tm2, N]

# ----------------------------
# Control inputs
# U = [P0_1, P0_2]
# ----------------------------
@variables P01(t) P02(t)
U = [P01, P02]

# ----------------------------
# Disturbances
# V = [PL1, PL2]
# ----------------------------
@variables PL1(t) PL2(t)
V = [PL1, PL2]

# ----------------------------
# Measurements
# Y = [PGm1, PGm2, F12]
# ----------------------------
@variables PGm1(t) PGm2(t)
F12 = KL*(θ1 - θ2)

Y = [PGm1,
    PGm2,
    F12
]
# Y = C X + E V + W
# E = [1 0;
#      0 1;
#      0 0]

f_eqs = [

    # dθ1/dt = ω1
    D(θ1) ~ ω1,

    # dω1/dt
    D(ω1) ~ -(KL/(J1*ω0))*θ1
             -(D1/J1)*ω1
             + (1/J1)*Tm1
             + (KL/(J1*ω0))*θ2,

    # dTm1/dt
    D(Tm1) ~ -β1*ω1
             -α1*ω0*Tm1
             -α1*Pr1*N,

    # dθ2/dt = ω2
    D(θ2) ~ ω2,

    # dω2/dt
    D(ω2) ~ +(KL/(J2*ω0))*θ1
             -(KL/(J2*ω0))*θ2
             -(D2/J2)*ω2
             + (1/J2)*Tm2,

    # dTm2/dt
    D(Tm2) ~ -β2*ω2
             -α2*ω0*Tm2
             -α2*Pr2*N,

    # dN/dt
    D(N) ~ -Ks*(J1/(J1+J2))*ω1
           -Ks*(J2/(J1+J2))*ω2
]

7-element Vector{Equation}:
 Differential(t, 1)(θ1(t)) ~ ω1(t)
 Differential(t, 1)(ω1(t)) ~ (KL*θ2(t)) / (J1*ω0) + Tm1(t) / J1 + (-KL*θ1(t)) / (J1*ω0) + (-D1*ω1(t)) / J1
 Differential(t, 1)(Tm1(t)) ~ -ω1(t)*β1 - Pr1*N(t)*α1 - Tm1(t)*α1*ω0
 Differential(t, 1)(θ2(t)) ~ ω2(t)
 Differential(t, 1)(ω2(t)) ~ (-D2*ω2(t)) / J2 + Tm2(t) / J2 + (-KL*θ2(t)) / (J2*ω0) + (KL*θ1(t)) / (J2*ω0)
 Differential(t, 1)(Tm2(t)) ~ -ω2(t)*β2 - Pr2*N(t)*α2 - Tm2(t)*α2*ω0
 Differential(t, 1)(N(t)) ~ (-J1*Ks*ω1(t)) / (J1 + J2) + (-J2*Ks*ω2(t)) / (J1 + J2)

In [6]:
function calcul_matrice_observabilite(f_eqs, y_eqs, states)
    println("1. Calcul des Jacobiennes A et C...")
    # A = df/dx
    A = Symbolics.jacobian(f_eqs, states)
    # C = dY/dx
    C = Symbolics.jacobian(y_eqs, states)
    
    n = length(states)
    
    # Initialisation de la matrice O avec C
    O = C
    
    # Puissance courante de A (A^0, A^1, A^2...)
    A_power = I(n) # Matrice identité initiale (si besoin) ou on commence la boucle direct
    
    println("2. Construction de la matrice O (CA^k)...")
    
    # Terme courant CA^k
    current_term = C
    
    # Boucle pour empiler C, CA, CA^2 ... CA^(n-1)
    # On a déjà C, donc on itère n-1 fois
    for i in 1:(n-1)
        # On calcule le terme suivant : C * A^i
        # Pour optimiser, on peut faire : Terme_precedent * A
        # Cependant, avec Symbolics, multiplier des matrices énormes peut être lourd.
        # On recalcule A^i ou on propage. Propager est plus simple :
        
        current_term = current_term * A
        
        # Concaténation verticale (vcat)
        O = [O; current_term]
    end
    
    return O, A, C
end

calcul_matrice_observabilite (generic function with 1 method)